**GOOGLE PLAYSTORE GAME SEARCH**

 Import all library needed to run the program, we'll use pandas, numpy, sklearn, and nltk.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer     #sklearn is used as media to run TF-IDF math.
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from nltk.stem.porter import PorterStemmer                      #nltk is used for tokenization and stemming.
from nltk.tokenize import word_tokenize
import nltk
from tabulate import tabulate
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**PREPROCESSING**

Read the entire csv dataset and remove all missing values and data duplicate 

In [2]:
df = pd.read_csv('googleplaystore.csv')

# Preprocessing
df = df.dropna(subset=['App'])                # Remove all missing values within app's column.
df = df.drop_duplicates(subset=['App'])       # Remove all duplicate and doubles.

stemmer = PorterStemmer()

In [3]:
def preprocess_text(text):
    tokens = word_tokenize(text)
    stemmed_tokens = [stemmer.stem(token) for token in tokens]    # Stemming to return all word into its base form.
    return ' '.join(stemmed_tokens)                               # Combining token stem into a single string with spaces.

In [4]:
df['text'] = df['App'].str.lower().apply(preprocess_text)         # Converts texts into lower case string
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df['text'])
tfidf_matrix = normalize(tfidf_matrix)                            # Turns every vector in 1 long string to do TF-IDF.

In [5]:
# Function to search for apps using GVSM and show similarity percentages
def search_app(query, similarity_threshold=0.1):
    # Preprocess query yet to be fill
    query = preprocess_text(query.lower())
    query_vec = vectorizer.transform([query])
    query_vec = normalize(query_vec)

    # Count the cosine similarity value using TF-IDF
    cosine_similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Remove all incorrect data 
    related_app_indices = np.where(cosine_similarities >= similarity_threshold)[0]

    # Get the similarity percentages
    similarity_percentages = cosine_similarities[related_app_indices] * 100
    sorted_indices = related_app_indices[np.argsort(-cosine_similarities[related_app_indices])]

    # Sort the cosine similarity percentages
    sorted_percentages = similarity_percentages[np.argsort(-cosine_similarities[related_app_indices])]
    results_df = df.iloc[sorted_indices][['App', 'Category']].copy()
    results_df['Similarity (%)'] = sorted_percentages

    return results_df

In [6]:
# Example usage
query = "Clash of clans"                                               # Query input
results = search_app(query)

print(f"Search results for '{query}':")
print(tabulate(results, headers='keys', tablefmt='psql'))     # Show table.


Search results for 'Clash of clans':
+-------+------------------------------------------------+---------------------+------------------+
|       | App                                            | Category            |   Similarity (%) |
|-------+------------------------------------------------+---------------------+------------------|
|  1670 | Clash of Clans                                 | GAME                |         100      |
|  7776 | Stats CR Clan for Clash Royale                 | FAMILY              |          66.8423 |
|  7525 | Battle of Zombies: Clans War                   | FAMILY              |          46.9915 |
|  1660 | Clash Royale                                   | GAME                |          45.1246 |
|  7784 | Helper for Clash Royale (All-in-1)             | FAMILY              |          35.6778 |
|  7777 | Ultimate Clash Royale Tracker                  | FAMILY              |          33.8716 |
|  7770 | Quiz for Clash Royale™                         | FAMI

In [7]:
import pandas as pd
from tabulate import tabulate

In [8]:
df = pd.read_csv('googleplaystore.csv')

# Preprocessing
df = df.dropna(subset=['App', 'Rating', 'Category', 'Type'])
df = df.drop_duplicates(subset=['App'])
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df = df.dropna(subset=['Rating'])

In [9]:
# A recommendation function based on rating, category, and type
def recommend_apps(rating_str, category, app_type, top_n=10):
    # Convert rating string ke float
    try:
        rating = float(rating_str)
    except ValueError:
        return pd.DataFrame()

    # Measurig the similarity based on the absolute value of the rating
    df['Similarity (%)'] = 100 - (abs(df['Rating'] - rating) * 20)

    # Filter apps by category and type
    target_apps = df[(df['Category'].str.contains(category, case=False)) &
                     (df['Type'] == app_type) &
                     (df['Similarity (%)'] >= 0)]

    # Sort by similarity percentage and select the top_n results
    target_apps = target_apps.sort_values(by='Similarity (%)', ascending=False).head(top_n)

    return target_apps[['App', 'Category', 'Rating', 'Type', 'Similarity (%)']]

In [10]:
# Example usage
target_rating_str = "4.7"
target_category = "Game"
target_type = "Paid"          # can be change between Paid/Free
recommendations = recommend_apps(target_rating_str, target_category, target_type)

print(f"Recommendations for {target_type.lower()} apps with rating {target_rating_str} in category {target_category}:")
print(tabulate(recommendations, headers='keys', tablefmt='psql'))

Recommendations for paid apps with rating 4.7 in category Game:
+-------+---------------------------+------------+----------+--------+------------------+
|       | App                       | Category   |   Rating | Type   |   Similarity (%) |
|-------+---------------------------+------------+----------+--------+------------------|
|  8879 | Riptide GP: Renegade      | GAME       |      4.7 | Paid   |              100 |
|  9024 | Retro City Rampage DX     | GAME       |      4.7 | Paid   |              100 |
|  8085 | Cytus II                  | GAME       |      4.7 | Paid   |              100 |
|  5648 | Five Nights at Freddy's 3 | GAME       |      4.7 | Paid   |              100 |
| 10060 | An Elite Warrior Ex       | GAME       |      4.7 | Paid   |              100 |
|  6895 | BW-Go                     | GAME       |      4.8 | Paid   |               98 |
|  6341 | Blackjack Verite Drills   | GAME       |      4.6 | Paid   |               98 |
|  5645 | Five Nights at Freddy's 4 